# 🧠 Introduction to NLP: From Text to Intelligence

**Welcome!** In this tutorial, we'll learn how computers understand human language. We'll go step by step, starting from the very basics.

By the end, you'll understand:
1. **Tokenisation** — How computers break text into pieces
2. **Stemming** — Chopping words down to their root
3. **Lemmatisation** — Finding the proper dictionary form of a word
4. **Embeddings** — How computers turn words into numbers (and why)
5. **Word Arithmetic** — The famous King − Man + Woman = Queen trick
6. **Semantic Search** — Finding similar meaning, not just matching keywords

---

### 💡 The Big Picture

Computers only understand numbers. They don't know what "cat" or "happy" means. So the entire field of NLP (Natural Language Processing) is about **converting human language into numbers** in clever ways that preserve meaning.

Think of it like translation: we're translating from *human language* into *maths* — and the better our translation, the smarter the computer seems.

---
## Part 1: Tokenisation 🔪

### What is tokenisation?

**Tokenisation** is the very first step in NLP. It means **breaking text into smaller pieces** called **tokens**.

Think of it like cutting a sentence into words — but it's actually more nuanced than that.

There are different ways to tokenise:

| Method | Example for "I'm running" | Tokens |
|--------|---------------------------|--------|
| **Word-level** | Split on spaces/punctuation | `["I", "'m", "running"]` |
| **Character-level** | Every character is a token | `["I", "'", "m", " ", "r", "u", "n", ...]` |
| **Subword (BPE)** | Splits rare words into pieces | `["I", "'m", " running"]` |

Modern AI models (like GPT) use **subword tokenisation** (called BPE — Byte Pair Encoding). Let's see all three in action!

### 1.1 Simple Word Tokenisation

The simplest approach: just split on spaces and punctuation.

In [ ]:
# The simplest tokeniser: just split on spaces
sentence = "The cat sat on the mat."
simple_tokens = sentence.split()
print("Simple split:", simple_tokens)
print(f"Number of tokens: {len(simple_tokens)}")

In [ ]:
# But notice the problem — "mat." includes the full stop!
# NLTK's word tokeniser is smarter:
import nltk
from nltk.tokenize import word_tokenize

sentence = "The cat sat on the mat. It didn't move!"
tokens = word_tokenize(sentence)
print("NLTK tokens:", tokens)
print(f"Number of tokens: {len(tokens)}")
print()
print("Notice how NLTK:")
print("  • Separated the full stop from 'mat'")
print("  • Split \"didn't\" into ['did', \"n't\"]")
print("  • Separated the exclamation mark")

### 1.2 Subword Tokenisation (BPE) — How GPT Does It

Modern language models like GPT-2 don't use word-level tokenisation. They use **Byte Pair Encoding (BPE)**.

**Why?** Because:
- There are too many unique words ("running", "runs", "runner", "ran" are all separate words!)
- What about new words the model has never seen? ("COVID", "blockchain")
- BPE finds a middle ground: common words stay whole, rare words get split into pieces

Let's see how GPT-2's tokeniser works:

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Let's tokenise the same sentence
text = "The cat sat on the mat."
token_ids = tokenizer.encode(text)
tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("Text:      ", text)
print("Token IDs: ", token_ids)
print("Tokens:    ", tokens)
print(f"\nThe model sees {len(token_ids)} numbers, not words!")
print("\nThe 'Ġ' symbol means 'this token starts with a space'")

In [ ]:
# Let's see what happens with unusual/rare words
examples = [
    "Hello world",
    "Supercalifragilisticexpialidocious",
    "The electroencephalograph machine beeped",
    "My name is Alex and I like Python programming",
    "🎉 Emojis work too!",
]

for text in examples:
    tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
    print(f"\n'{text}'")
    print(f"  → {tokens}")
    print(f"  → {len(tokens)} tokens")

### 🔑 Key Insight: The Vocabulary

Every tokeniser has a fixed **vocabulary** — a dictionary mapping tokens to numbers.

- GPT-2 has a vocabulary of **50,257** tokens
- Each token gets a unique ID number
- The model never sees text — it only sees these numbers!

This is like giving the computer a codebook: "whenever you see the number 464, that means 'The'".

In [ ]:
print(f"GPT-2 vocabulary size: {tokenizer.vocab_size:,} tokens")
print()

# Let's look up some specific words
for word in ["hello", " hello", "the", " the", "cat", " Python"]:
    ids = tokenizer.encode(word)
    print(f"  '{word}' → ID {ids} → tokens {tokenizer.convert_ids_to_tokens(ids)}")

print("\n💡 Notice: ' the' (with space) and 'the' (no space) are DIFFERENT tokens!")

### ✏️ Try It Yourself!

Change the text below and see how GPT-2 tokenises it. Try your own name, a German word, some technical jargon — anything!

In [ ]:
# 👇 CHANGE THIS TEXT and re-run the cell!
my_text = "Your text goes here"

ids = tokenizer.encode(my_text)
tokens = tokenizer.convert_ids_to_tokens(ids)
print(f"Text: '{my_text}'")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"IDs: {ids}")

---
## Part 2: Stemming ✂️

### What is stemming?

**Stemming** is a crude but fast way to reduce words to their **root form** by chopping off endings.

It's like taking a shortcut: instead of understanding the grammar, just cut off common suffixes like "-ing", "-ed", "-tion".

| Word | Stem |
|------|------|
| running | run |
| happily | happili |
| connection | connect |
| better | better |

⚠️ **Notice**: stemming doesn't always produce real words! "happily" → "happili" is not a real word. That's OK — it's just for grouping similar words together.

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")

words = ["running", "runs", "ran", "runner",
         "happily", "happiness", "happy",
         "connection", "connected", "connecting",
         "better", "best", "good",
         "studies", "studying", "studied"]

print(f"{'Word':<15} {'Porter':<15} {'Snowball':<15}")
print("-" * 45)
for word in words:
    print(f"{word:<15} {porter.stem(word):<15} {snowball.stem(word):<15}")

### 🤔 Stemming Limitations

Look at the results above:
- ✅ "running", "runs", "runner" → all map to "run" — great!
- ❌ "better" and "good" don't map to the same stem (they should — "better" is the comparative of "good")
- ❌ "happily" → "happili" — not a real word

Stemming is **fast** but **dumb**. It doesn't understand grammar or meaning. That's where lemmatisation comes in...

---
## Part 3: Lemmatisation 📖

### What is lemmatisation?

**Lemmatisation** is the smarter cousin of stemming. Instead of blindly chopping suffixes, it uses a **dictionary and grammar rules** to find the proper base form of a word (called the **lemma**).

| Word | Stem (crude) | Lemma (smart) |
|------|------|-------|
| running | run | run |
| better | better | good |
| mice | mice | mouse |
| was | wa | be |
| happily | happili | happily |

The key difference: **lemmas are always real words**.

In [ ]:
import spacy

# spaCy is a powerful NLP library that does lemmatisation (and much more)
nlp = spacy.load("en_core_web_sm")

sentence = "The mice were running quickly towards the better cheese. She was happily studying."
doc = nlp(sentence)

print(f"{'Word':<15} {'Lemma':<15} {'Part of Speech':<15}")
print("-" * 45)
for token in doc:
    if not token.is_punct and not token.is_space:
        print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<15}")

In [ ]:
# Let's compare stemming vs lemmatisation side by side
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

test_words = ["running", "better", "mice", "was", "went", "children", "happily", "studies", "geese", "ate"]

print(f"{'Word':<15} {'Stem':<15} {'Lemma':<15}")
print("-" * 45)
for word in test_words:
    stem = stemmer.stem(word)
    lemma = nlp(word)[0].lemma_
    match = "✅" if stem == lemma else "❌"
    print(f"{word:<15} {stem:<15} {lemma:<15} {match}")

print("\n💡 Notice how lemmatisation correctly handles irregular forms:")
print("   'went' → 'go', 'mice' → 'mouse', 'ate' → 'eat'")
print("   Stemming can't do this — it only chops off endings.")

### Stemming vs Lemmatisation — When to Use Which?

| | Stemming | Lemmatisation |
|---|---------|---------------|
| **Speed** | ⚡ Very fast | 🐢 Slower |
| **Accuracy** | Approximate | Precise |
| **Output** | May not be real words | Always real words |
| **Use case** | Search engines, quick indexing | Chatbots, text analysis |
| **Needs** | Just rules | Dictionary + grammar model |

In modern NLP with transformer models (GPT, BERT etc.), we often skip both — the model learns its own internal representations. But understanding these concepts is still crucial for knowing **why** those models work.

---
## Part 4: Word Embeddings 🌌

### This is the most important part!

Everything we've done so far (tokenisation, stemming, lemmatisation) is just **preprocessing**. Now we get to the real magic.

### The Problem: Computers Need Numbers

We could assign each word a number: cat=1, dog=2, house=3. But this tells the computer that "dog" is between "cat" and "house" — which is meaningless!

### The Solution: Embeddings

An **embedding** represents each word as a **list of numbers** (called a **vector**). Not just one number — many numbers!

```
"cat"   → [0.2, -0.4, 0.7, 0.1, ...]
"dog"   → [0.3, -0.3, 0.6, 0.2, ...]
"house" → [-0.8, 0.1, -0.2, 0.9, ...]
```

The magic: **words with similar meanings get similar numbers!**

"cat" and "dog" have similar vectors (both are animals), while "house" has a very different vector.

### What are "Dimensions"?

The number of values in each vector is called the **dimension** (or **embedding dimension**).

| Model | Dimensions | What it means |
|-------|-----------|---------------|
| GloVe (small) | **50** | Each word = 50 numbers |
| Word2Vec | **300** | Each word = 300 numbers |
| GPT-2 (small) | **768** | Each token = 768 numbers |
| GPT-4 | **~12,288** | Each token = ~12,288 numbers |

More dimensions = more nuance the model can capture, but also more memory and computation.

Think of it like describing a person:
- **2 dimensions**: height, weight → rough description
- **50 dimensions**: add hair colour, age, shoe size, etc. → much better
- **768 dimensions**: every conceivable detail → incredibly precise

In [ ]:
# Let's load pre-trained word vectors (GloVe — trained on Wikipedia)
# These are 50-dimensional: each word is represented by 50 numbers
import gensim.downloader as api
import numpy as np

print("Loading GloVe word vectors (50 dimensions)...")
glove = api.load("glove-wiki-gigaword-50")
print(f"Loaded! Vocabulary size: {len(glove):,} words")
print(f"Embedding dimension: {glove.vector_size}")

In [ ]:
# Let's look at what an embedding actually looks like
word = "cat"
vector = glove[word]

print(f"The word '{word}' is represented as {len(vector)} numbers:")
print()
print("[", end="")
for i, val in enumerate(vector):
    if i > 0:
        print(",", end="")
    if i % 10 == 0:
        print(f"\n  ", end="")
    print(f"{val:7.4f}", end="")
print("\n]")
print(f"\n💡 These {len(vector)} numbers encode everything the model knows about '{word}'.")
print(f"   The numbers were learned by reading billions of words of text.")

### 4.1 Similar Words Have Similar Vectors

The whole point of embeddings is that **meaning is encoded in the numbers**. Words that appear in similar contexts ("cat" and "dog" both appear near "pet", "fur", "animal") end up with similar vectors.

We measure similarity using **cosine similarity** — a number between -1 and 1:
- **1.0** = identical meaning
- **0.0** = completely unrelated
- **-1.0** = opposite meaning

In [ ]:
# Find the most similar words to a given word
for word in ["cat", "king", "computer", "happy", "germany"]:
    similar = glove.most_similar(word, topn=5)
    neighbours = ", ".join([f"{w} ({s:.2f})" for w, s in similar])
    print(f"'{word}' is most similar to: {neighbours}")
    print()

In [ ]:
# Let's measure similarity between specific pairs
pairs = [
    ("cat", "dog"),        # both animals
    ("cat", "kitten"),     # same animal, different age
    ("cat", "car"),        # similar spelling, different meaning
    ("cat", "democracy"),  # completely unrelated
    ("happy", "sad"),      # opposites
    ("happy", "joyful"),   # synonyms
    ("king", "queen"),     # related concepts
    ("paris", "france"),   # city and country
]

print(f"{'Word 1':<15} {'Word 2':<15} {'Similarity':>10}")
print("-" * 42)
for w1, w2 in pairs:
    sim = glove.similarity(w1, w2)
    bar = "█" * int(abs(sim) * 20)
    print(f"{w1:<15} {w2:<15} {sim:>8.3f}  {bar}")

---
## Part 5: Word Arithmetic 👑

### The Famous King − Man + Woman = Queen

This is the most mind-blowing property of word embeddings.

Because words are just lists of numbers (vectors), we can **do maths with them**!

The idea:
- Take the vector for "king"
- Subtract the vector for "man" (remove the "maleness")
- Add the vector for "woman" (add "femaleness")
- The result is closest to... **"queen"**! 🎉

This means the model has learned the **concept of gender** just from reading text — nobody told it that kings are male or queens are female!

In [ ]:
# The classic: king - man + woman = ?
result = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=5)
print("king − man + woman = ?")
print()
for word, score in result:
    marker = " 👑" if word == "queen" else ""
    print(f"  {word:<15} (similarity: {score:.4f}){marker}")

In [ ]:
# Let's try more royal analogies!
analogies = [
    # [word1, word2, word3] → word1 - word2 + word3 = ?
    (["queen", "woman"], ["man"], "queen − woman + man"),
    (["prince", "man"], ["woman"], "prince − man + woman"),
    (["king", "prince"], ["princess"], "king − prince + princess"),
    (["princess", "woman"], ["man"], "princess − woman + man"),
]

for positive, negative, description in analogies:
    result = glove.most_similar(positive=positive, negative=negative, topn=3)
    answers = ", ".join([f"{w} ({s:.3f})" for w, s in result])
    print(f"{description} → {answers}")

In [ ]:
# It works for other relationships too!
analogies = [
    # Countries and capitals
    (["paris", "germany"], ["france"], "paris − france + germany"),
    (["tokyo", "france"], ["japan"], "tokyo − japan + france"),
    
    # Verb tenses
    (["walking", "swam"], ["walked"], "walking − walked + swam"),
    
    # Comparatives
    (["bigger", "cold"], ["big"], "bigger − big + cold"),
    
    # Plurals  
    (["cars", "child"], ["car"], "cars − car + child"),
]

print("🌍 Capitals, tenses, plurals — it all works!\n")
for positive, negative, description in analogies:
    result = glove.most_similar(positive=positive, negative=negative, topn=3)
    answers = ", ".join([f"{w} ({s:.3f})" for w, s in result])
    print(f"{description}")
    print(f"  → {answers}\n")

### 🧮 Let's Visualise the Maths

The vectors are 50-dimensional (impossible to visualise directly), but we can project them down to 2D to get an intuition:

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# Royal family words
words = ["king", "queen", "prince", "princess", "man", "woman", "boy", "girl"]
vectors = np.array([glove[w] for w in words])

# Project 50 dimensions → 2 dimensions for plotting
pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.scatter(coords[:, 0], coords[:, 1], s=100, c=['royalblue', 'crimson', 'dodgerblue', 'hotpink',
                                                    'steelblue', 'salmon', 'lightblue', 'lightpink'])
for i, word in enumerate(words):
    ax.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=14, fontweight='bold',
                xytext=(8, 8), textcoords='offset points')

# Draw arrows showing the gender direction
for m, f in [("king", "queen"), ("prince", "princess"), ("man", "woman"), ("boy", "girl")]:
    mi, fi = words.index(m), words.index(f)
    ax.annotate("", xy=(coords[fi, 0], coords[fi, 1]),
                xytext=(coords[mi, 0], coords[mi, 1]),
                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5, ls="--"))

ax.set_title("Word Embeddings: Royal Family\n(50D projected to 2D)", fontsize=16)
ax.set_xlabel(f"Component 1 (explains {pca.explained_variance_ratio_[0]:.0%} of variance)")
ax.set_ylabel(f"Component 2 (explains {pca.explained_variance_ratio_[1]:.0%} of variance)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Notice how the arrows (male → female) all point in roughly the same direction!")
print("   The model has learned a 'gender direction' in its 50-dimensional space.")

In [ ]:
# Let's also visualise a broader set of words — animals, countries, emotions
word_groups = {
    "Animals 🐾": ["cat", "dog", "fish", "bird", "horse", "cow"],
    "Countries 🌍": ["france", "germany", "japan", "china", "brazil", "australia"],
    "Emotions 😊": ["happy", "sad", "angry", "fear", "love", "hate"],
    "Food 🍕": ["pizza", "pasta", "sushi", "bread", "rice", "cheese"],
}

all_words = []
all_colors = []
color_map = {"Animals 🐾": "green", "Countries 🌍": "blue", "Emotions 😊": "red", "Food 🍕": "orange"}

for group, words in word_groups.items():
    all_words.extend(words)
    all_colors.extend([color_map[group]] * len(words))

vectors = np.array([glove[w] for w in all_words])
pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(12, 9))
for group, color in color_map.items():
    mask = [c == color for c in all_colors]
    ax.scatter(coords[mask, 0], coords[mask, 1], c=color, s=100, label=group, alpha=0.7)

for i, word in enumerate(all_words):
    ax.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=11, xytext=(5, 5), textcoords='offset points')

ax.set_title("Word Embeddings: Different Categories Cluster Together!", fontsize=16)
ax.legend(fontsize=12, loc="best")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Words from the same category cluster together in embedding space!")
print("   The model learned these groupings just from reading text — no labels needed.")

---
## Part 6: How the Lookup Works 🔍

### The Embedding Table

Under the hood, an embedding is just a **big lookup table** (a matrix).

If you have:
- Vocabulary of 50,257 tokens (GPT-2)
- Embedding dimension of 768

Then the embedding table is a matrix with **50,257 rows × 768 columns**.

```
Token ID 0    → [0.12, -0.34, 0.56, ..., 0.89]   (768 numbers)
Token ID 1    → [0.45, 0.23, -0.67, ..., 0.12]   (768 numbers)
Token ID 2    → [-0.78, 0.91, 0.34, ..., -0.56]  (768 numbers)
...           → ...
Token ID 50256 → [0.33, -0.12, 0.78, ..., 0.45]  (768 numbers)
```

When the model sees token ID 464, it just looks up **row 464** in the table. That's it! No computation — just a table lookup.

These numbers are **learned during training** — the model adjusts them so that similar words end up with similar vectors.

In [ ]:
# Let's look at GPT-2's actual embedding table
from transformers import GPT2Model
import torch

model = GPT2Model.from_pretrained("gpt2")
embedding_table = model.wte.weight.data  # wte = Word Token Embeddings

print(f"GPT-2 Embedding Table Shape: {embedding_table.shape}")
print(f"  → {embedding_table.shape[0]:,} tokens (vocabulary size)")
print(f"  → {embedding_table.shape[1]} dimensions per token")
print(f"  → Total parameters: {embedding_table.shape[0] * embedding_table.shape[1]:,}")
print(f"  → Memory: {embedding_table.shape[0] * embedding_table.shape[1] * 4 / 1e6:.1f} MB (in float32)")

In [ ]:
# The lookup in action — let's trace how "Hello world" becomes numbers
text = "Hello world"

# Step 1: Tokenise → get token IDs
token_ids = tokenizer.encode(text)
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(f"Step 1 — Tokenise: '{text}' → {tokens} → IDs {token_ids}")
print()

# Step 2: Look up each ID in the embedding table
print("Step 2 — Lookup each ID in the embedding table:")
for tid, tok in zip(token_ids, tokens):
    vector = embedding_table[tid]
    print(f"  Token '{tok}' (ID {tid}) → [{vector[0]:.4f}, {vector[1]:.4f}, {vector[2]:.4f}, ... , {vector[-1]:.4f}]")
    print(f"     (768 numbers — showing first 3 and last 1)")

print()
print(f"📊 Result: '{text}' ({len(token_ids)} tokens) becomes a {len(token_ids)} × 768 matrix")
print(f"   This matrix is what GPT-2 actually processes!")

### Different Models, Different Dimensions

Every model has its own embedding size. Bigger models use more dimensions to capture more nuance:

| Model | Embedding Dimensions | Parameters |
|-------|---------------------|------------|
| GloVe (small) | 50 | - |
| GloVe (large) | 300 | - |
| BERT (base) | 768 | 110M |
| GPT-2 (small) | 768 | 124M |
| GPT-2 (large) | 1,280 | 774M |
| GPT-3 | 12,288 | 175B |
| GPT-4 | ~12,288+ | ~1.8T |

**Key insight**: You can't mix embeddings from different models! GPT-2's 768-dimensional vector for "cat" is completely different from BERT's 768-dimensional vector for "cat". They were trained separately and live in different "spaces".

In [ ]:
# Let's visualise how the SAME word looks in different embedding dimensions
word = "cat"

# GloVe 50D
glove_vec = glove[word]

# GPT-2 768D
cat_id = tokenizer.encode(word)[0]
gpt2_vec = embedding_table[cat_id].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(len(glove_vec)), glove_vec, color='steelblue', width=1.0)
axes[0].set_title(f"GloVe embedding for '{word}'\n({len(glove_vec)} dimensions)", fontsize=13)
axes[0].set_xlabel("Dimension")
axes[0].set_ylabel("Value")

axes[1].bar(range(len(gpt2_vec)), gpt2_vec, color='coral', width=1.0)
axes[1].set_title(f"GPT-2 embedding for '{word}'\n({len(gpt2_vec)} dimensions)", fontsize=13)
axes[1].set_xlabel("Dimension")
axes[1].set_ylabel("Value")

plt.suptitle(f"Same word, different models = different representations", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f"💡 GloVe uses {len(glove_vec)} numbers to describe '{word}'")
print(f"   GPT-2 uses {len(gpt2_vec)} numbers — 15× more detail!")
print(f"   But you can't compare them — they're in completely different spaces.")

---
## Part 7: Semantic Search 🔎

### Beyond Keyword Matching

Traditional search (like Ctrl+F) matches **exact keywords**. If you search for "automobile", you won't find documents that say "car".

**Semantic search** uses embeddings to find results based on **meaning**, not just exact words.

How it works:
1. Convert all your documents into embedding vectors
2. Convert the search query into an embedding vector
3. Find the documents whose vectors are most similar to the query vector

This is how modern search engines, recommendation systems, and RAG (Retrieval-Augmented Generation) work!

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Our "document database" — imagine these are real documents
documents = [
    "The cat sat on the warm mat by the fireplace",
    "Dogs are loyal and friendly pets that love walks",
    "Python is a popular programming language for AI",
    "The stock market crashed due to economic concerns",
    "She drove her new automobile to the office today",
    "Machine learning models need lots of training data",
    "The chef prepared a delicious Italian pasta dish",
    "Climate change is affecting weather patterns globally",
    "The kitten played with a ball of yarn all afternoon",
    "Neural networks are inspired by the human brain",
    "He bought a fast car with excellent fuel efficiency",
    "The puppy wagged its tail excitedly at the visitor",
]

def embed_sentence(sentence, model):
    """Simple sentence embedding: average the word vectors."""
    words = sentence.lower().split()
    vectors = [model[w] for w in words if w in model]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Pre-compute embeddings for all documents
doc_embeddings = np.array([embed_sentence(doc, glove) for doc in documents])

print(f"Embedded {len(documents)} documents into {doc_embeddings.shape[1]}-dimensional vectors")
print(f"Embedding matrix shape: {doc_embeddings.shape}")

In [ ]:
def semantic_search(query, doc_embeddings, documents, model, top_k=5):
    """Search documents by meaning, not keywords."""
    query_vec = embed_sentence(query, model).reshape(1, -1)
    similarities = cosine_similarity(query_vec, doc_embeddings)[0]
    
    ranked = sorted(enumerate(similarities), key=lambda x: -x[1])
    
    print(f"🔎 Query: '{query}'\n")
    for rank, (idx, sim) in enumerate(ranked[:top_k]):
        bar = "█" * int(sim * 30)
        print(f"  {rank+1}. [{sim:.3f}] {bar}")
        print(f"     {documents[idx]}\n")

# Search for "vehicle" — should find car/automobile docs even though "vehicle" isn't in them!
semantic_search("vehicle", doc_embeddings, documents, glove)

In [ ]:
# Search for "cute pet" — should find cat/dog/kitten/puppy docs
semantic_search("cute pet", doc_embeddings, documents, glove)

In [ ]:
# Search for "artificial intelligence" — should find ML/neural network docs
semantic_search("artificial intelligence", doc_embeddings, documents, glove)

In [ ]:
# Now let's compare: keyword search vs semantic search
query = "automobile"

print("=" * 60)
print("KEYWORD SEARCH (exact match)")
print("=" * 60)
keyword_results = [doc for doc in documents if query.lower() in doc.lower()]
if keyword_results:
    for doc in keyword_results:
        print(f"  ✅ {doc}")
else:
    print(f"  ❌ No results for '{query}'")

# Only 1 document contains "automobile"
print()
keyword_results2 = [doc for doc in documents if "car" in doc.lower()]
print(f"  (Keyword 'car' finds {len(keyword_results2)} result(s) separately)")

print()
print("=" * 60)
print("SEMANTIC SEARCH (meaning-based)")
print("=" * 60)
semantic_search(query, doc_embeddings, documents, glove, top_k=3)

print("💡 Semantic search finds BOTH 'automobile' AND 'car' documents!")
print("   It understands they mean the same thing.")

### ✏️ Try Your Own Searches!

Change the query below and see what comes up:

In [ ]:
# 👇 CHANGE THIS QUERY!
my_query = "cooking food"

semantic_search(my_query, doc_embeddings, documents, glove)

---
## Part 8: Putting It All Together 🏗️

Let's recap the full NLP pipeline — from raw text to intelligent search:

```
Raw Text
  │
  ▼
Tokenisation (break into pieces)
  │
  ▼
Normalisation: Stemming / Lemmatisation (reduce to base forms)
  │
  ▼
Embedding Lookup (convert tokens → vectors of numbers)
  │
  ▼
AI Model processes the numbers (GPT, BERT, etc.)
  │
  ▼
Output: translation, summary, answer, search results...
```

### Key Takeaways:

1. **Tokenisation** breaks text into pieces the model can handle
   - Modern models use subword tokenisation (BPE)
   - Each model has its own tokeniser and vocabulary

2. **Stemming** is fast but crude — just chops off endings
   - "running" → "run", but "better" stays "better"

3. **Lemmatisation** is smart but slower — uses grammar knowledge
   - "better" → "good", "mice" → "mouse"

4. **Embeddings** are the magic — turning words into meaningful numbers
   - Similar words → similar vectors
   - The dimension (50, 768, 12288) determines how much detail is captured
   - Learned from massive amounts of text

5. **Word arithmetic** proves embeddings capture real meaning
   - King − Man + Woman = Queen
   - This works for countries, tenses, plurals, and more

6. **Semantic search** finds results by meaning, not keywords
   - This is the foundation of modern AI search and RAG systems

In [ ]:
# Final demo: the complete pipeline on one sentence
from nltk.stem import PorterStemmer

text = "The quick brown foxes were jumping over the lazy dogs"
print(f"Original: {text}")
print()

# 1. Tokenise
tokens_nltk = word_tokenize(text)
print(f"1. Tokenised (NLTK):  {tokens_nltk}")

tokens_gpt2 = tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
print(f"   Tokenised (GPT-2): {tokens_gpt2}")
print()

# 2. Stem
stemmer = PorterStemmer()
stemmed = [stemmer.stem(t) for t in tokens_nltk]
print(f"2. Stemmed:   {stemmed}")
print()

# 3. Lemmatise
doc = nlp(text)
lemmas = [token.lemma_ for token in doc if not token.is_punct]
print(f"3. Lemmatised: {lemmas}")
print()

# 4. Embed
sentence_vec = embed_sentence(text, glove)
print(f"4. Sentence embedding: [{sentence_vec[0]:.4f}, {sentence_vec[1]:.4f}, ..., {sentence_vec[-1]:.4f}]")
print(f"   ({len(sentence_vec)} dimensions)")
print()

# 5. Find similar sentences
print("5. Most similar documents:")
semantic_search(text, doc_embeddings, documents, glove, top_k=3)

---
## 🎓 Bonus: Explore On Your Own

Here are some things to try:

1. **Word analogies**: Try `glove.most_similar(positive=["berlin", "france"], negative=["germany"])` — what do you get?

2. **Odd one out**: `glove.doesnt_match(["breakfast", "lunch", "dinner", "computer"])` — which word doesn't belong?

3. **Add your own documents** to the search database and try queries

4. **Compare tokenisers**: How does GPT-2 tokenise German vs English? Try `tokenizer.encode("Donaudampfschifffahrtsgesellschaftskapitän")`

In [ ]:
# Playground — try things here!

# Odd one out
print("Odd one out:")
for group in [["breakfast", "lunch", "dinner", "computer"],
              ["cat", "dog", "horse", "airplane"],
              ["red", "blue", "green", "table"]]:
    odd = glove.doesnt_match(group)
    print(f"  {group} → '{odd}' doesn't belong!")

print()

# German compound word
german = "Donaudampfschifffahrtsgesellschaftskapitän"
gpt_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(german))
print(f"GPT-2 tokenises '{german}' into {len(gpt_tokens)} pieces:")
print(f"  {gpt_tokens}")
print()
print("💡 BPE splits this monster word into recognisable sub-pieces!")